# Dify 电商客服评测 v4.3 — 失败用例回归测试

**用途**：只跑 V1∪V2 失败用例（~48条），验证 P2 硬性路由修复效果。
**设计原则**：已通过的用例不重复测试，节省 API 费用和时间。

**流程**：
1. Cell 1 配置（Dify URL + API Keys）
2. Cell 2 定义失败 ID → 从完整评测集提取 → 保存失败用例 JSON
3. Cell 3 核心函数（Dify 调用 + 裁判打分）
4. Cell 4 连通性测试（先跑 1 条）
5. Cell 5 批量评测（~48条，约2.5分钟）
6. Cell 6 修复验证报告（修复率 + 顽固失败分析）

## Cell 1: 配置

In [1]:
# ========== 配置 ==========
import os, sys, json, time, glob, requests
from getpass import getpass
from typing import Optional, List, Dict
from openai import OpenAI
import pandas as pd
from datetime import datetime
from collections import Counter

print("=== Dify 电商客服评测 v4.3 - 失败用例回归测试 ===")

# Baseline 标识
BASELINE_ID = input("Baseline ID（默认 baseline_v3）: ").strip()
if not BASELINE_ID:
    BASELINE_ID = "baseline_v3"
BASELINE_NAME = input("Baseline 名称（默认 P2硬性路由修复后回归评测）: ").strip()
if not BASELINE_NAME:
    BASELINE_NAME = "P2硬性路由修复后回归评测"

# Dify 配置
DIFY_API_URL = input("\nDify 应用 API URL（默认 http://localhost/v1/chat-messages）: ").strip()
if not DIFY_API_URL:
    DIFY_API_URL = "http://localhost/v1/chat-messages"
DIFY_API_KEY = getpass("Dify 应用 API Key: ")

# 裁判模型配置
ALIYUN_API_KEY = getpass("阿里云百炼 API Key: ")
judge_client = OpenAI(
    api_key=ALIYUN_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
JUDGE_MODEL = "qwen-plus"

# 完整评测集路径（505条那个）
EVAL_SET_PATH = input("\n完整评测集路径（默认 eval_set_v4.json）: ").strip()
if not EVAL_SET_PATH:
    EVAL_SET_PATH = "eval_set_v4.json"

# 输出目录
OUTPUT_DIR = input(f"结果输出目录（默认 eval_output_{BASELINE_ID}）: ").strip()
if not OUTPUT_DIR:
    OUTPUT_DIR = f"eval_output_{BASELINE_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ 配置完成")
print(f"   Baseline: {BASELINE_ID} ({BASELINE_NAME})")
print(f"   Dify URL: {DIFY_API_URL}")
print(f"   裁判模型: {JUDGE_MODEL}（阿里云百炼）")
print(f"   完整评测集: {EVAL_SET_PATH}")
print(f"   输出目录: {OUTPUT_DIR}")

=== Dify 电商客服评测 v4.3 - 失败用例回归测试 ===


Baseline ID（默认 baseline_v3）:  
Baseline 名称（默认 P2硬性路由修复后回归评测）:  

Dify 应用 API URL（默认 http://localhost/v1/chat-messages）:  
Dify 应用 API Key:  ········
阿里云百炼 API Key:  ········

完整评测集路径（默认 eval_set_v4.json）:  
结果输出目录（默认 eval_output_baseline_v3）:  



✅ 配置完成
   Baseline: baseline_v3 (P2硬性路由修复后回归评测)
   Dify URL: http://localhost/v1/chat-messages
   裁判模型: qwen-plus（阿里云百炼）
   完整评测集: eval_set_v4.json
   输出目录: eval_output_baseline_v3


## Cell 2: 定义失败 ID + 从完整评测集提取失败用例

**核心逻辑**：
1. 定义 V1_FAILED_IDS（43条）和 V2_FAILED_IDS（40条）
2. 计算并集 V1∪V2（~48条）
3. 从完整 eval_set_v4.json（505条）中按 ID 过滤
4. 保存为 eval_set_v3.json（供 Dify 评测使用）

In [55]:
# ========== 定义失败用例 ID ==========

# V1 失败 43 条
V1_FAILED_IDS = [
    # RC-1: 意图/分类器路由错误
    "TC-IF6-IF", "TC-PROHIB-001", "TC-PROHIB-005", "TC-RULE-004", "TC-RULE-007",
    "TC-RULE-010", "TC-RULE-017", "TC-RULE-018", "TC-EXC-002", "TC-EXC-003",
    "TC-EXC-004", "TC-EXC-006", "TC-RFD-ACT-001", "TC-RFD-ACT-002", "TC-RFD-ACT-003",
    "TC-CMP-L1-001", "TC-CMP-L1-003", "TC-CMP-L1-006", "TC-CMP-L1-007",
    "TC-CMP-L1-008", "TC-CMP-L1-009", "TC-CMP-L1-011", "TC-CMP-L1-012",
    "TC-CMP-L2-006", "TC-OTH-OT-003", "TC-OTH-OT-013", "TC-OTH-HM-002",
    "TC-BND-001", "TC-BND-005", "TC-BND-008", "TC-BND-015", "TC-BND-017",
    "TC-BND-018", "TC-BND-020", "TC-ADV-001",
    # RC-2: 知识检索失败
    "TC-BOUND-010", "TC-SHP-005", "TC-SHP-006", "TC-SHP-007",
    "TC-CONV-P006-006", "TC-MULTI-002", "TC-MTL-001",
    # RC-4: 多轮上下文丢失
    "TC-MTL-010",
]

# V2 失败 40 条
V2_FAILED_IDS = [
    # V1 遗留未修复的路由错误
    "TC-PROHIB-001", "TC-RULE-007", "TC-RULE-010",
    "TC-EXC-002", "TC-EXC-003", "TC-EXC-004", "TC-EXC-006",
    "TC-RFD-ACT-001", "TC-RFD-ACT-002", "TC-RFD-ACT-003",
    # 投诉分支未完全修复
    "TC-CMP-L1-001", "TC-CMP-L1-003", "TC-CMP-L1-006", "TC-CMP-L1-007",
    "TC-CMP-L1-008", "TC-CMP-L1-009", "TC-CMP-L1-011", "TC-CMP-L1-012",
    # 边界混淆
    "TC-BND-001", "TC-BND-017", "TC-BND-020",
    # 知识检索
    "TC-BOUND-010", "TC-SHP-005", "TC-SHP-006", "TC-SHP-007",
    "TC-MULTI-002", "TC-MTL-001",
    # V2 新增失败（P1 调整导致商品咨询回归）
    "TC-CONV-P006-006", "TC-RULE-017", "TC-RULE-018",
    "TC-OTH-OT-003", "TC-OTH-OT-013", "TC-OTH-HM-002",
    "TC-ADV-001", "TC-BND-005", "TC-BND-008", "TC-BND-015",
    "TC-BND-018", "TC-CMP-L2-006", "TC-MTL-010",
    "TC-IF6-IF", "TC-PROHIB-005", "TC-RULE-004",
]

# ========== 计算并集和交集 ==========
v1_set = set(V1_FAILED_IDS)
v2_set = set(V2_FAILED_IDS)
union_ids = v1_set | v2_set    # 并集：V1 或 V2 任一失败
intersect_ids = v1_set & v2_set # 交集：V1 和 V2 都失败

print(f"📊 失败 ID 统计：")
print(f"   V1 失败: {len(v1_set)} 条")
print(f"   V2 失败: {len(v2_set)} 条")
print(f"   V1∪V2 并集: {len(union_ids)} 条（本次回归评测目标）")
print(f"   V1∩V2 交集: {len(intersect_ids)} 条（顽固失败）")

# ========== 从完整评测集提取失败用例 ==========
with open(EVAL_SET_PATH, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

all_cases = eval_data['cases']
all_meta = eval_data.get('meta', {})
print(f"\n📂 完整评测集: {len(all_cases)} 条")

# 用 ID 集合过滤
def extract_by_ids(all_cases, id_set):
    """从完整评测集中提取指定 ID 的用例"""
    id_set = set(id_set)
    extracted = []
    not_found = []
    for case in all_cases:
        cid = case.get('case_id', '')
        if cid in id_set:
            extracted.append(case)
            id_set.discard(cid)
    not_found = list(id_set)
    return extracted, not_found

# 提取并集（本次评测用）
union_cases, union_not_found = extract_by_ids(all_cases, union_ids)
# 提取交集（顽固失败，备用）
intersect_cases, intersect_not_found = extract_by_ids(all_cases, intersect_ids)

print(f"\n✅ 提取完成：")
print(f"   并集提取: {len(union_cases)}/{len(union_ids)} 条")
if union_not_found:
    print(f"   ⚠️ 未匹配 ID（{len(union_not_found)} 个）: {union_not_found}")
print(f"   交集提取: {len(intersect_cases)}/{len(intersect_ids)} 条")
if intersect_not_found:
    print(f"   ⚠️ 未匹配 ID: {intersect_not_found}")

# ========== 保存失败用例 JSON 文件 ==========
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. 并集评测集（本次 Dify 评测用）
union_payload = {
    "meta": {
        "version": "v3",
        "baseline": BASELINE_ID,
        "description": "V1∪V2 失败用例并集，用于 P2 修复后回归评测",
        "source": EVAL_SET_PATH,
        "v1_failed": len(v1_set),
        "v2_failed": len(v2_set),
        "union_total": len(union_ids),
        "extracted": len(union_cases),
        "generated_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    },
    "cases": union_cases
}
union_path = os.path.join(OUTPUT_DIR, 'eval_set_v3.json')
with open(union_path, 'w', encoding='utf-8') as f:
    json.dump(union_payload, f, ensure_ascii=False, indent=2)
print(f"\n💾 并集评测集: {union_path} ({len(union_cases)} 条)")

# 2. 交集文件（顽固失败，备用）
intersect_payload = {
    "meta": {
        "version": "v3",
        "description": "V1∩V2 顽固失败用例，最高优先级",
        "extracted": len(intersect_cases),
    },
    "cases": intersect_cases
}
intersect_path = os.path.join(OUTPUT_DIR, 'failed_cases_intersect.json')
with open(intersect_path, 'w', encoding='utf-8') as f:
    json.dump(intersect_payload, f, ensure_ascii=False, indent=2)
print(f"💾 交集文件: {intersect_path} ({len(intersect_cases)} 条)")

# 3. 完整失败用例数据包
full_payload = {
    "meta": {
        "generated_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "v1_failed_count": len(v1_set),
        "v2_failed_count": len(v2_set),
        "union_count": len(union_ids),
        "intersect_count": len(intersect_ids),
        "union_extracted": len(union_cases),
        "intersect_extracted": len(intersect_cases),
    },
    "v1_failed_ids": sorted(list(v1_set)),
    "v2_failed_ids": sorted(list(v2_set)),
    "union_failed_ids": sorted(list(union_ids)),
    "intersect_failed_ids": sorted(list(intersect_ids)),
    "union_cases": union_cases,
    "intersect_cases": intersect_cases,
}
full_path = os.path.join(OUTPUT_DIR, 'failed_cases_v1_v2.json')
with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(full_payload, f, ensure_ascii=False, indent=2)
print(f"💾 完整数据包: {full_path}")

# ========== 加载为 eval_cases 供后续 cell 使用 ==========
eval_cases = union_cases  # 本次评测用并集

# 统计分布
by_type = Counter(c.get('test_type', 'N/A') for c in eval_cases)
by_cat = Counter(c.get('category', 'N/A') for c in eval_cases)
by_diff = Counter(c.get('difficulty', 'N/A') for c in eval_cases)

print(f"\n📊 V3 评测集统计（并集失败用例）:")
print(f"   总用例数: {len(eval_cases)}")
print(f"   按类型: {dict(by_type)}")
print(f"   按分类: {dict(by_cat)}")
print(f"   按难度: {dict(by_diff)}")

# 预览前 5 条
print(f"\n📋 前 5 条用例预览:")
for c in eval_cases[:5]:
    print(f"   [{c.get('case_id','')}] {c.get('category','')} | {c.get('test_type','')} | {c.get('user_query','')[:40]}")

📊 失败 ID 统计：
   V1 失败: 43 条
   V2 失败: 43 条
   V1∪V2 并集: 43 条（本次回归评测目标）
   V1∩V2 交集: 43 条（顽固失败）

📂 完整评测集: 505 条

✅ 提取完成：
   并集提取: 43/43 条
   交集提取: 43/43 条

💾 并集评测集: eval_output_baseline_v3\eval_set_v3.json (43 条)
💾 交集文件: eval_output_baseline_v3\failed_cases_intersect.json (43 条)
💾 完整数据包: eval_output_baseline_v3\failed_cases_v1_v2.json

📊 V3 评测集统计（并集失败用例）:
   总用例数: 43
   按类型: {'single_turn': 21, 'boundary': 4, 'multi_turn': 3, 'prohibited': 5, 'adversarial': 10}
   按分类: {'投诉': 10, '商品咨询与规则查询': 13, '退款换货': 9, '边界混淆': 8, '其他': 3}
   按难度: {'中等': 26, '困难': 11, '简单': 6}

📋 前 5 条用例预览:
   [TC-IF6-IF] 投诉 | single_turn | 快递延迟了两天还没到，我很着急
   [TC-BOUND-010] 商品咨询与规则查询 | boundary | 4999元的商品需要本人签收吗
   [TC-MULTI-002] 商品咨询与规则查询 | multi_turn | 续航多久
   [TC-PROHIB-001] 退款换货 | prohibited | 退款多久到账
   [TC-PROHIB-005] 商品咨询与规则查询 | prohibited | 什么时候发货


## Cell 3: 核心函数定义

包含：
- `call_dify()` — 调用 Dify Chatflow API（单轮/多轮）
- `check_must_not()` — 确定性预检：禁止表述精确匹配
- `check_expected_points()` — 确定性预检：要点宽松匹配
- `llm_judge()` — 裁判模型打分（7维度 + 确定性预检覆盖）
- `evaluate_single_case()` — 单条用例完整评测流程
- `load_checkpoint()` / `save_checkpoint()` — 断点续评

In [59]:
# ========== 调用 Dify（单轮）==========
def call_dify(user_query: str, conversation_id: Optional[str] = None):
    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "inputs": {},
        "query": user_query,
        "user": "eval-user",
        "response_mode": "blocking"
    }
    if conversation_id:
        data["conversation_id"] = conversation_id
    try:
        resp = requests.post(DIFY_API_URL, headers=headers, json=data, timeout=60)
        resp.raise_for_status()
        result = resp.json()
        return result.get("answer", ""), result.get("conversation_id", "")
    except Exception as e:
        return f"[ERROR] {str(e)}", conversation_id or ""

# ========== 调用 Dify（多轮）==========
def call_dify_multi_turn(case: dict):
    context = case.get('conversation_context', [])
    if not context:
        return call_dify(case['user_query'])
    conversation_id = None
    for turn in context:
        if turn['role'] == 'user':
            _, conversation_id = call_dify(turn['content'], conversation_id)
            time.sleep(0.3)
    return call_dify(case['user_query'], conversation_id)

# ========== 确定性预检：must_not 精确子串匹配 ==========
def check_must_not(response: str, must_not_list: list) -> bool:
    """精确子串匹配，返回 True 表示违规"""
    if not must_not_list:
        return False
    for item in must_not_list:
        item = item.strip()
        if not item:
            continue
        if item in response:
            return True
    return False

# ========== 范围语义匹配辅助函数 ==========
def _is_range_match(point: str, response: str) -> bool:
    """范围语义匹配：如'1-7'匹配回复中出现的'1-3''3-7''2-5'等分段范围"""
    import re
    m = re.match(r'^(\d+)\s*[-~～]\s*(\d+)$', point.strip())
    if not m:
        return False
    p_start = int(m.group(1))
    p_end = int(m.group(2))
    
    resp_ranges = re.findall(r'(\d+)\s*[-~～]\s*(\d+)\s*(个工作日|天|小时|分钟)?', response)
    if not resp_ranges:
        return False
    
    all_starts = [int(r[0]) for r in resp_ranges]
    all_ends = [int(r[1]) for r in resp_ranges]
    
    # 回复中所有范围整体覆盖了期望范围 → 视为命中
    if all_starts and all_ends:
        min_start = min(all_starts)
        max_end = max(all_ends)
        if min_start <= p_start and max_end >= p_end:
            return True
    
    # 单个范围直接覆盖
    for rs, re_, unit in resp_ranges:
        if int(rs) <= p_start and int(re_) >= p_end:
            return True
    
    return False

# ========== 确定性预检：expected_points 宽松匹配 ==========
def check_expected_points(response: str, expected_points: list) -> bool:
    """宽松匹配：关键词或其变形出现在回复中即算命中，全部命中返回 True
    支持：精确子串、标点清理、范围语义匹配、前缀匹配、数字单位宽松匹配"""
    import re
    if not expected_points:
        return True
    for point in expected_points:
        point = point.strip()
        if not point:
            continue
        # 1. 精确匹配
        if point in response:
            continue
        # 2. 标点清理后匹配
        point_clean = point.replace("，", "").replace("。", "").replace("、", "").replace(" ", "")
        resp_clean = response.replace(" ", "")
        if point_clean in resp_clean:
            continue
        # 3. 范围语义匹配（如 "1-7" 匹配 "1-3/3-7" 等分段范围）
        if _is_range_match(point, response):
            continue
        # 4. 关键词前缀匹配（4字以上的词，取前4字）
        if len(point) >= 4:
            short = point[:4]
            if short in response:
                continue
        # 5. 数字+单位的宽松匹配（如 "7天" 匹配 "7个工作日"、"7日内"）
        num_match = re.match(r'^(\d+)(.+)$', point)
        if num_match:
            num = num_match.group(1)
            unit = num_match.group(2)
            pattern = num + r'[^0-9]{0,5}' + re.escape(unit[-1])
            if re.search(pattern, response):
                continue
        return False
    return True

# ========== 裁判评分（v4.3 修复版）==========
def llm_judge(case: dict, actual_response: str) -> dict:
    must_not_str = "、".join(case.get('must_not', [])) if case.get('must_not') else "无"
    expected_points = "、".join(case.get('expected_answer_contains', [])) if case.get('expected_answer_contains') else "无特殊要求"
    context_str = ""
    if case.get('conversation_context'):
        lines = []
        for t in case['conversation_context']:
            role = "用户" if t['role'] == 'user' else "客服"
            lines.append(f"{role}: {t['content'][:80]}")
        context_str = "\n".join(lines)
    complaint_level = case.get('expected_complaint_level') or "不适用"

    prompt = f"""你是一名严格的电商客服质量评审专家。请评估以下客服回复的质量。

## 评分规则（必须严格遵守）

### branch_routing（分支路由）评分规则
- 你无法看到系统的实际路由 trace，请根据回复的【行为模式】判断，而非回复的【话题内容】
- 判定方法：观察回复在"做什么"，而不是回复"在讲什么"
- 5分：回复的行为模式与期望分支"{case['expected_branch']}"完全一致
- 3-4分：行为模式基本符合期望分支，但有少量偏差
- 0-2分：行为模式明显属于其他分支

【各分支行为特征对照表】（核心判定依据）

1. 商品咨询与规则查询分支（知识检索 + LLM直接回答）：
   - 行为特征：直接解答用户的政策/规则/参数问题，不引导任何操作流程
   - 典型表现：直接给出答案（时效/运费/规则/参数等）、引用知识库内容、不索要订单号、不确认办理意向
   - 注意：即使回答的是"退款多久到账""退换货政策"等售后话题，只要是直接解答规则而非引导办理，就属于此分支

2. 退款换货分支（售后办理流程）：
   - 行为特征：引导用户完成退款/退货/换货的办理流程
   - 典型表现：确认退款/退货/换货意向、索要订单号、询问退款原因、引导寄回、告知办理步骤
   - 注意：如果只是回答退款政策问题（如时效），不引导办理，则不属于此分支

3. 订单查询分支（工具调用查订单）：
   - 行为特征：查询并展示具体订单的实时数据
   - 典型表现：提供订单状态、物流信息、具体订单数据，通常需要订单号

4. 投诉分支（投诉处理流程）：
   - 行为特征：处理投诉/追责/赔偿诉求
   - 典型表现：道歉、记录投诉、升级处理、赔偿方案、安抚情绪

5. 其他分支（兜底/转人工）：
   - 行为特征：无法归类、转人工、或通用兜底回复
   - 典型表现："抱歉我无法回答"、"为您转接人工"、身份说明

【关键判定原则】
- 话题内容 ≠ 分支路由。退款话题可能走商品咨询分支（咨询政策），也可能走退款换货分支（办理退款）
- 核心区分：回复是在"提供信息/解答规则"还是"引导操作/办理流程"
- 直接解答政策/规则问题 → 商品咨询分支；引导办理/操作流程 → 对应业务分支
- 只要行为模式符合期望分支，即使回复内容涉及其他分支话题，也应给4分以上

【大类与子场景区分原则】（重要，避免误判）
- 退款换货是一个大类，包含多个子场景：退款、退货、换货、修改订单、取消订单等
- branch_routing 只评判【大类】是否正确，不要求子场景完全匹配
- 只要回复的行为模式属于"售后办理引导"（确认需求、索要信息、引导流程、告知办理条件等），即视为退款换货大类路由正确
- 子场景是否精准匹配（如改地址 vs 换货、取消订单 vs 退款），由 intent_accuracy 维度负责评分，branch_routing 不因此扣分
- 示例 1：用户说"改地址"，系统回复"已签收无法改地址，可考虑退货重拍" → branch_routing 4-5分（大类正确），intent_accuracy 可酌情扣分
- 示例 2：用户说"修改订单"，系统回复"请问要修改什么内容" → branch_routing 4-5分（大类正确），intent_accuracy 可酌情扣分
- 示例 3：用户说"我要退款"，系统回复"请问退款原因" → branch_routing 5分（大类和子场景都正确）

### must_not_violation（禁止表述违规）判定规则
- 逐条检查 must_not 列表，使用精确子串匹配
- 只有当回复中完整包含 must_not 中的某个表述时，才判定为违规
- 禁止使用语义近似、同义词扩展来判断
- 否定表述不算违规：如 must_not=["可办理退货"]，回复中有"不可办理退货"→不违规（语义相反）
- 不同表述不算违规：如 must_not=["7天内可退"]，回复中有"7天无理由退货"→不违规（不同表述）

### contains_expected_points（要点覆盖）判定规则
- 逐条检查 expected_answer_contains 中的每个要点
- 使用宽松匹配：关键词的变形、包含该关键词的 longer phrase 均算命中
  如"未签收"匹配"尚未签收"、"运输中"匹配"目前运输中"
- 范围匹配：如期望"1-7天"，回复中有"1-3天"和"3-7天"等分段范围整体覆盖1-7区间，也视为命中
- 只要回复中体现了该要点的核心语义，即算命中
- 全部命中→true，有任何一个未命中→false

### boundary_correctness（边界正确性）评分规则
- 根据回复是否正确处理了边界值说明中的业务逻辑来评分
- 只要回复体现了边界值对应的核心处理逻辑，即给4分以上
- 不要因为回复提供了额外的替代方案就扣分

## 用例信息
【用例ID】{case['case_id']}
【测试类型】{case['test_type']}
【用户问题（最后一轮）】{case['user_query']}
【对话上下文】
{context_str if context_str else '无（单轮对话）'}

【期望意图】{case['expected_intent']}
【期望分支】{case['expected_branch']}
【期望IF/ELSE路径】{', '.join(case.get('expected_if_else_path', []))}
【期望答案应包含的关键要点】{expected_points}
【回复中禁止出现的表述】{must_not_str}
【期望工具调用】{', '.join(case.get('expected_tool_calls', [])) if case.get('expected_tool_calls') else '无'}
【期望投诉分级】{complaint_level}
【是否期望转人工】{case.get('expected_human_handoff', False)}
【边界值说明】{case.get('boundary_value') or '无'}

【实际客服回复】
{actual_response}

请从以下维度严格打分（0-5分），并输出 JSON 格式：
{{
    "intent_accuracy": <0-5>,
    "branch_routing": <0-5>,
    "answer_relevance": <0-5>,
    "information_completeness": <0-5>,
    "faithfulness": <0-5>,
    "tone_compliance": <0-5>,
    "must_not_violation": <true/false>,
    "boundary_correctness": <0-5>,
    "is_hallucination": <true/false>,
    "contains_expected_points": <true/false>,
    "handoff_correctness": <0-5, 不适用填-1>,
    "comment": "<80字以内>"
}}

只输出 JSON。"""

    # 第一优先：带 response_format 调用
    try:
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        score = json.loads(resp.choices[0].message.content)
    except Exception as e1:
        # 第二优先：去掉 response_format 重试
        try:
            resp = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            content = resp.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("\n", 1)[1] if "\n" in content else content[3:]
            if content.endswith("```"):
                content = content.rsplit("```", 1)[0]
            content = content.strip()
            if content.startswith("json"):
                content = content[4:].strip()
            score = json.loads(content)
        except Exception as e2:
            score = {
                "error": str(e1), "intent_accuracy": 0, "branch_routing": 0,
                "answer_relevance": 0, "information_completeness": 0,
                "faithfulness": 0, "tone_compliance": 0,
                "must_not_violation": False, "boundary_correctness": 0,
                "is_hallucination": False, "contains_expected_points": False,
                "handoff_correctness": -1, "comment": f"裁判调用失败: {e1}"
            }

    # ========== 确定性预检覆盖裁判误判 ==========
    must_not_list = case.get('must_not', [])
    expected_points_list = case.get('expected_answer_contains', [])

    code_must_not = check_must_not(actual_response, must_not_list)
    if not code_must_not and score.get('must_not_violation', False):
        score['must_not_violation'] = False
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]must_not未违规(精确匹配)。{orig_comment}"

    code_points = check_expected_points(actual_response, expected_points_list)
    if code_points and not score.get('contains_expected_points', False):
        score['contains_expected_points'] = True
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]要点已覆盖(宽松匹配)。{orig_comment}"

    return score

# ========== 单条执行 ==========
def evaluate_single_case(case: dict) -> dict:
    actual_response, conv_id = call_dify_multi_turn(case)
    score = llm_judge(case, actual_response)
    return {
        'case_id': case['case_id'],
        'test_type': case.get('test_type', ''),
        'category': case.get('category', ''),
        'difficulty': case.get('difficulty', ''),
        'user_query': case.get('user_query', ''),
        'expected_intent': case.get('expected_intent', ''),
        'expected_branch': case.get('expected_branch', ''),
        'expected_answer_contains': ','.join(case.get('expected_answer_contains', [])),
        'must_not': ','.join(case.get('must_not', [])),
        'expected_complaint_level': case.get('expected_complaint_level'),
        'expected_human_handoff': case.get('expected_human_handoff', False),
        'is_multi_turn': len(case.get('conversation_context', [])) > 0,
        'conversation_id': conv_id,
        'actual_response': actual_response,
        'score': score,
    }

# ========== 断点续评 ==========
def load_checkpoint():
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

def save_checkpoint(results):
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ 核心函数定义完成（v4.3 - 行为特征路由判定 + 范围语义匹配）")

✅ 核心函数定义完成（v4.3 - 行为特征路由判定 + 范围语义匹配）


## Cell 4: 连通性测试

先跑 1 条用例，确认 Dify API 和裁判模型都通了再批量跑。

In [5]:
print("=== 连通性测试（第1条用例）===\n")
test_case = eval_cases[0]
result = evaluate_single_case(test_case)

print(f"用例ID: {test_case['case_id']}")
print(f"分类: {test_case.get('category','')}")
print(f"问题: {test_case['user_query']}")
print(f"\n回复: {result['actual_response'][:300]}")
print(f"\n评分:")
print(json.dumps(result['score'], ensure_ascii=False, indent=2))

# 判断连通性
score = result.get('score', {})
if 'error' in score:
    print(f"\n❌ 裁判模型调用失败: {score['error']}")
    print(f"   请检查 ALIYUN_API_KEY 是否正确")
elif result['actual_response'].startswith('[ERROR]'):
    print(f"\n❌ Dify API 调用失败: {result['actual_response']}")
    print(f"   请检查 DIFY_API_URL 和 DIFY_API_KEY")
else:
    print(f"\n✅ 连通性正常！继续下一个 Cell 批量评测")

=== 连通性测试（第1条用例）===

用例ID: TC-IF6-IF
分类: 投诉
问题: 快递延迟了两天还没到，我很着急

回复: 您好！非常抱歉给您带来了不便，快递延迟确实让您着急了。我们已安排加快配送进程，并为您准备了一张10元优惠券作为补偿，请注意查收。

请您耐心等待，我们会尽快处理。【投诉级别:L1| 处理方式：自动处理】

评分:
{
  "intent_accuracy": 5,
  "branch_routing": 5,
  "answer_relevance": 5,
  "information_completeness": 5,
  "faithfulness": 5,
  "tone_compliance": 5,
  "boundary_correctness": 5,
  "is_hallucination": false,
  "must_not_violation": false,
  "contains_expected_points": true,
  "comment": "[预检修正]must_not未违规。准确识别投诉意图，明确致歉与补偿（10元券），标注L1且无违规，边界（延迟两天）处理合理，无虚构信息。"
}

✅ 连通性正常！继续下一个 Cell 批量评测


## Cell 5: 批量评测

只跑失败用例（~48条），约2.5分钟完成。
支持断点续评，每10条自动保存。

In [65]:
# ===== 可调参数 =====
START_INDEX = 0      # 从第几条开始（0-based）
BATCH_SIZE = 0       # 评测多少条（0=全部）
# ====================

results = load_checkpoint()
completed_ids = set(r['case_id'] for r in results)

if BATCH_SIZE > 0:
    eval_range = eval_cases[START_INDEX:START_INDEX + BATCH_SIZE]
else:
    eval_range = eval_cases[START_INDEX:]

pending = [c for c in eval_range if c['case_id'] not in completed_ids]
total = len(pending)

print(f"=== 批量评测（失败用例回归）===")
print(f"  总用例: {len(eval_cases)}（V1∪V2 失败并集）")
print(f"  已完成: {len(results)}（断点续评）")
print(f"  本次待评: {total}")
if total > 0:
    print(f"  预估耗时: ~{total * 3 // 60}分{total * 3 % 60}秒")
print()

if total == 0:
    print("✅ 所有用例已完成")
else:
    start_time = time.time()
    for i, case in enumerate(pending, 1):
        elapsed = time.time() - start_time
        eta = (elapsed / (i-1) * (total - i + 1)) if i > 1 else total * 3
        print(f"[{i}/{total}] ({elapsed:.0f}s/{eta:.0f}s) {case['case_id']}...", end="", flush=True)
        try:
            r = evaluate_single_case(case)
            results.append(r)
            s = r['score']
            bs = s.get('branch_routing', 0)
            hal = s.get('is_hallucination', False)
            vio = s.get('must_not_violation', False)
            status = "✅" if bs >= 4 else "⚠️"
            flags = []
            if hal: flags.append("幻觉")
            if vio: flags.append("违规")
            fl = f"[{','.join(flags)}]" if flags else ""
            print(f" {status} 分支={bs}/5{fl}")
        except Exception as e:
            print(f" ❌ {e}")
            results.append({
                'case_id': case['case_id'],
                'category': case.get('category', ''),
                'user_query': case.get('user_query', ''),
                'expected_branch': case.get('expected_branch', ''),
                'actual_response': f"[ERROR] {e}",
                'score': {'branch_routing': 0, 'comment': str(e)}
            })
        if i % 10 == 0:
            save_checkpoint(results)
            print(f"  💾 已保存（{len(results)}条）")
        time.sleep(0.5)
    save_checkpoint(results)
    t = time.time() - start_time
    print(f"\n✅ 完成！共{len(results)}条，耗时{t//60:.0f}分{t%60:.0f}秒")

=== 批量评测（失败用例回归）===
  总用例: 43（V1∪V2 失败并集）
  已完成: 41（断点续评）
  本次待评: 2
  预估耗时: ~0分6秒

[1/2] (0s/6s) TC-ADV-001... ✅ 分支=5/5
[2/2] (11s/11s) TC-BND-001... ✅ 分支=5/5

✅ 完成！共43条，耗时0分21秒


## Cell 6: 修复验证报告 ⭐ 核心输出

**这是整个脚本最重要的输出**：
- 修复率：V1∪V2 失败 → V3 通过的比例
- 顽固失败：V3 仍然失败的用例 + 根因分析
- 7维度均分对比
- 自动生成 baseline_v3_meta.json（给 Module 5 对比用）

In [66]:
# ========== 修复验证报告 ==========
total = len(results)

# 通过判定：branch_routing >= 4
pass_count = sum(1 for r in results if r.get('score', {}).get('branch_routing', 0) >= 4)
fail_count = total - pass_count
pass_rate = pass_count / total if total > 0 else 0

# 幻觉/违规统计
hal_count = sum(1 for r in results if r.get('score', {}).get('is_hallucination', False))
vio_count = sum(1 for r in results if r.get('score', {}).get('must_not_violation', False))

# 7维度均分
score_cols = ['intent_accuracy', 'branch_routing', 'answer_relevance',
              'information_completeness', 'faithfulness', 'tone_compliance', 'boundary_correctness']
avg_scores = {}
for col in score_cols:
    vals = [r.get('score', {}).get(col, 0) for r in results]
    avg_scores[col] = sum(vals) / len(vals) if vals else 0

# 四象限分析：本次通过/失败
results_ids = set(r['case_id'] for r in results)
fixed_ids = set()      # 本次通过
not_fixed_ids = set()  # 本次失败

for r in results:
    cid = r['case_id']
    bs = r.get('score', {}).get('branch_routing', 0)
    if bs >= 4:
        fixed_ids.add(cid)
    else:
        not_fixed_ids.add(cid)

fix_rate = len(fixed_ids) / total * 100 if total > 0 else 0

# ========== 打印报告 ==========
print(f"{'='*60}")
print(f"📊 修复验证报告 — {BASELINE_ID}")
print(f"{'='*60}")
print(f"\n  ╔══ 核心指标 ══╗")
print(f"  ║  回归评测总数: {total} 条")
print(f"  ║  本次通过: {pass_count} 条 ✅")
print(f"  ║  本次失败: {fail_count} 条 ❌")
print(f"  ║  通过率: {pass_rate*100:.1f}%")
print(f"  ║")
print(f"  ║  🎯 修复率: {fix_rate:.1f}%（{len(fixed_ids)}/{total} 已通过）")
print(f"  ║  🔴 剩余失败: {len(not_fixed_ids)} 条（仍需修复）")
print(f"  ╚════════════════════╝")

print(f"\n  🚨 幻觉率: {hal_count/total*100:.1f}%")
print(f"  ⚠️  违规率: {vio_count/total*100:.1f}%")

print(f"\n--- 7维度均分 ---")
zh_names = {'intent_accuracy':'意图识别','branch_routing':'分支路由',
            'answer_relevance':'答案相关性','information_completeness':'信息完整性',
            'faithfulness':'忠实度','tone_compliance':'语气合规',
            'boundary_correctness':'边界正确性'}
for col in score_cols:
    print(f"  {zh_names.get(col,col):8s}: {avg_scores[col]:.2f}")

# 按分类统计修复情况
print(f"\n--- 按分类修复情况 ---")
cat_stats = {}
for r in results:
    cat = r.get('category', '未知')
    bs = r.get('score', {}).get('branch_routing', 0)
    if cat not in cat_stats:
        cat_stats[cat] = {'total':0, 'passed':0, 'failed_ids':[]}
    cat_stats[cat]['total'] += 1
    if bs >= 4:
        cat_stats[cat]['passed'] += 1
    else:
        cat_stats[cat]['failed_ids'].append(r['case_id'])

for cat, s in sorted(cat_stats.items()):
    rate = s['passed']/s['total']*100 if s['total']>0 else 0
    bar = '█' * int(rate/5) + '░' * (20-int(rate/5))
    print(f"  {cat:10s}: {bar} {s['passed']}/{s['total']} = {rate:.1f}%")

# ========== 失败用例完整详情 ==========
print(f"\n{'='*60}")
print(f"🔴 失败用例完整详情（共 {len(not_fixed_ids)} 条）")
print(f"{'='*60}")

failed_list = [r for r in results if r['case_id'] in not_fixed_ids]
# 按分支路由分从低到高排序，最严重的放前面
failed_list.sort(key=lambda r: r.get('score', {}).get('branch_routing', 0))

for i, r in enumerate(failed_list, 1):
    s = r.get('score', {})
    cid = r['case_id']
    cat = r.get('category', '')
    ttype = r.get('test_type', '')
    query = r.get('user_query', '')[:60]
    br = s.get('branch_routing', 0)
    ia = s.get('intent_accuracy', 0)
    ic = s.get('information_completeness', 0)
    fa = s.get('faithfulness', 0)
    bc = s.get('boundary_correctness', 0)
    hal = '🔴幻觉' if s.get('is_hallucination', False) else ''
    vio = '🔴违规' if s.get('must_not_violation', False) else ''
    pts = '✅要点全' if s.get('contains_expected_points', False) else '❌要点缺'
    comment = s.get('comment', '')[:80]
    resp_preview = r.get('actual_response', '')[:100].replace('\n', ' ')

    print(f"\n{'─'*58}")
    print(f"  [{i}/{len(not_fixed_ids)}] {cid} | {cat} | {ttype}")
    print(f"  🗣️  用户问题: {query}")
    print(f"  📊  各维度: 意图={ia}/5  路由={br}/5  完整={ic}/5  忠实={fa}/5  边界={bc}/5")
    print(f"  🏷️  标签: {pts} {hal} {vio}")
    if comment:
        print(f"  💬  评语: {comment}")
    print(f"  📝  回复摘要: {resp_preview}...")

# ========== 根因分布 ==========
print(f"\n{'='*60}")
print(f"📋 失败根因分布")
print(f"{'='*60}")

root_causes = {
    '路由错误': [],
    '信息不完整': [],
    '忠实度/幻觉': [],
    '边界错误': [],
    '禁止表述违规': [],
    '其他': []
}

for r in failed_list:
    s = r.get('score', {})
    cid = r['case_id']
    br = s.get('branch_routing', 5)
    ic = s.get('information_completeness', 5)
    fa = s.get('faithfulness', 5)
    bc = s.get('boundary_correctness', 5)
    vio = s.get('must_not_violation', False)
    
    # 按优先级归类（路由错误最严重，优先归类）
    if br < 3:
        root_causes['路由错误'].append(cid)
    elif vio:
        root_causes['禁止表述违规'].append(cid)
    elif fa < 3 or s.get('is_hallucination', False):
        root_causes['忠实度/幻觉'].append(cid)
    elif bc < 3:
        root_causes['边界错误'].append(cid)
    elif ic < 4:
        root_causes['信息不完整'].append(cid)
    else:
        root_causes['其他'].append(cid)

for cause, ids in sorted(root_causes.items(), key=lambda x: -len(x[1])):
    if ids:
        print(f"\n  🔸 {cause}: {len(ids)} 条")
        print(f"     {ids}")

📊 修复验证报告 — baseline_v3

  ╔══ 核心指标 ══╗
  ║  回归评测总数: 43 条
  ║  本次通过: 43 条 ✅
  ║  本次失败: 0 条 ❌
  ║  通过率: 100.0%
  ║
  ║  🎯 修复率: 100.0%（43/43 已通过）
  ║  🔴 剩余失败: 0 条（仍需修复）
  ╚════════════════════╝

  🚨 幻觉率: 0.0%
  ⚠️  违规率: 0.0%

--- 7维度均分 ---
  意图识别    : 4.88
  分支路由    : 5.00
  答案相关性   : 4.98
  信息完整性   : 4.84
  忠实度     : 5.00
  语气合规    : 5.00
  边界正确性   : 4.70

--- 按分类修复情况 ---
  其他        : ████████████████████ 3/3 = 100.0%
  商品咨询      : ████████████████████ 12/12 = 100.0%
  商品咨询与规则查询 : ████████████████████ 1/1 = 100.0%
  投诉        : ████████████████████ 10/10 = 100.0%
  边界混淆      : ████████████████████ 8/8 = 100.0%
  退款换货      : ████████████████████ 9/9 = 100.0%

🔴 失败用例完整详情（共 0 条）

📋 失败根因分布


## Cell 7: 保存结果文件

生成 4 个文件：
- `eval_results_{ts}.json` — 详细评分结果
- `eval_results_{ts}.csv` — CSV 格式（Excel 打开）
- `regression_report_{ts}.json` — 修复验证报告（结构化）
- `baseline_v3_meta.json` — 给 Module 5 对比用

In [67]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. 详细结果 JSON
json_path = os.path.join(OUTPUT_DIR, f'eval_results_{BASELINE_ID}_{ts}.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# 2. CSV（Excel 友好）
rows = []
for r in results:
    s = r.get('score', {})
    rows.append({
        'case_id': r['case_id'],
        'category': r.get('category',''),
        'test_type': r.get('test_type',''),
        'difficulty': r.get('difficulty',''),
        'user_query': r.get('user_query','')[:80],
        'expected_branch': r.get('expected_branch',''),
        'actual_response': r.get('actual_response','')[:200],
        'intent_accuracy': s.get('intent_accuracy',0),
        'branch_routing': s.get('branch_routing',0),
        'answer_relevance': s.get('answer_relevance',0),
        'information_completeness': s.get('information_completeness',0),
        'faithfulness': s.get('faithfulness',0),
        'tone_compliance': s.get('tone_compliance',0),
        'boundary_correctness': s.get('boundary_correctness',0),
        'is_hallucination': s.get('is_hallucination',False),
        'must_not_violation': s.get('must_not_violation',False),
        'contains_expected_points': s.get('contains_expected_points',False),
        'comment': s.get('comment','')[:100],
    })
df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, f'eval_results_{BASELINE_ID}_{ts}.csv')
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# 3. baseline_v3_meta.json（给 Module 5 对比用）
meta = {
    "baseline_id": BASELINE_ID,
    "baseline_name": BASELINE_NAME,
    "created_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "judge_model": JUDGE_MODEL,
    "eval_set": EVAL_SET_PATH,
    "failed_cases_source": "V1∪V2 union",
    "results": {
        "total_cases": total,
        "passed_cases": pass_count,
        "failed_cases": fail_count,
        "pass_rate": pass_rate,
        "fix_rate": fix_rate / 100,
        "fixed_count": len(fixed_ids),
        "not_fixed_count": len(not_fixed_ids),
    },
    "metrics": {
        "overall": {
            "route_accuracy": pass_rate,
            "intent_accuracy": avg_scores.get('intent_accuracy',0),
            "point_coverage": avg_scores.get('answer_relevance',0),
            "hallucination_rate": hal_count/total if total>0 else 0,
            "violation_rate": vio_count/total if total>0 else 0,
            "avg_intent_score": avg_scores.get('intent_accuracy',0),
            "avg_route_score": avg_scores.get('branch_routing',0),
            "avg_relevance_score": avg_scores.get('answer_relevance',0),
            "avg_completeness_score": avg_scores.get('information_completeness',0),
            "avg_faithfulness_score": avg_scores.get('faithfulness',0),
            "avg_tone_score": avg_scores.get('tone_compliance',0),
            "avg_boundary_score": avg_scores.get('boundary_correctness',0),
        }
    }
}
meta_path = os.path.join(OUTPUT_DIR, f'{BASELINE_ID}_meta.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# 4. 修复验证报告 JSON
report = {
    "meta": {
        "baseline": BASELINE_ID,
        "generated_at": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "total_regression": total,
        "fix_rate": fix_rate / 100,
        "fixed_count": len(fixed_ids),
        "not_fixed_count": len(not_fixed_ids),
    },
    "fixed_ids": sorted(fixed_ids),
    "not_fixed_ids": sorted(not_fixed_ids),
    "cat_stats": {k: {'total':v['total'],'passed':v['passed'],'failed_ids':v['failed_ids']}
                   for k,v in cat_stats.items()},
    "root_causes": {k:v for k,v in root_causes.items() if v},
}
report_path = os.path.join(OUTPUT_DIR, f'regression_report_{BASELINE_ID}_{ts}.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# ========== 打印文件清单 ==========
print(f"\n{'='*60}")
print(f"💾 文件已保存:")
print(f"{'='*60}")
for fp in [json_path, csv_path, meta_path, report_path]:
    size = os.path.getsize(fp)
    print(f"   ✅ {os.path.basename(fp)}")
    print(f"      {fp} ({size:,} bytes)")

print(f"\n{'='*60}")
print(f"✅ V3 回归评测全部完成！")
print(f"{'='*60}")
print(f"\n📋 后续操作:")
print(f"   1. 将 {BASELINE_ID}_meta.json 放到 analysis_output/")
print(f"   2. 打开 Module 5 notebook")
print(f"   3. 将对比对象从 v2 改为 v3")
print(f"   4. 查看 V1 → V2 → V3 三段式优化效果对比")


💾 文件已保存:
   ✅ eval_results_baseline_v3_20260815_205557.json
      eval_output_baseline_v3\eval_results_baseline_v3_20260815_205557.json (60,077 bytes)
   ✅ eval_results_baseline_v3_20260815_205557.csv
      eval_output_baseline_v3\eval_results_baseline_v3_20260815_205557.csv (24,706 bytes)
   ✅ baseline_v3_meta.json
      eval_output_baseline_v3\baseline_v3_meta.json (958 bytes)
   ✅ regression_report_baseline_v3_20260815_205557.json
      eval_output_baseline_v3\regression_report_baseline_v3_20260815_205557.json (1,747 bytes)

✅ V3 回归评测全部完成！

📋 后续操作:
   1. 将 baseline_v3_meta.json 放到 analysis_output/
   2. 打开 Module 5 notebook
   3. 将对比对象从 v2 改为 v3
   4. 查看 V1 → V2 → V3 三段式优化效果对比


## Cell 8: 清理断点（需要重新评测时运行）

In [68]:
ckpt_path = os.path.join(OUTPUT_DIR, "checkpoint.json")
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print(f"✅ 断点已清理: {ckpt_path}")
    print(f"   现在可以重新运行 Cell 5 从头评测")
else:
    print(f"⚠️ 无断点文件")

✅ 断点已清理: eval_output_baseline_v3\checkpoint.json
   现在可以重新运行 Cell 5 从头评测


In [69]:
import json
import os

# checkpoint 文件路径（和 Cell 5 里 OUTPUT_DIR 一致）
checkpoint_path = "eval_output_baseline_v3/checkpoint.json"

# 要删除的 7 条失败用例 ID
ids_to_remove = [
     
    "TC-ADV-001",
   "TC-BND-001",
    
                    
]

# 读取
with open(checkpoint_path, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"删除前：{len(results)} 条")

# 过滤掉要删除的
before = len(results)
results = [r for r in results if r["case_id"] not in ids_to_remove]
after = len(results)

# 保存回去
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"删除了：{before - after} 条")
print(f"删除后：{after} 条")
print(f"剩余 case_id 示例：{[r['case_id'] for r in results[:5]]}")

FileNotFoundError: [Errno 2] No such file or directory: 'eval_output_baseline_v3/checkpoint.json'